# Archive historique — preparer_evaluation_extraction

Ce notebook est conservé pour documenter la préparation et les évaluations antérieures. Il ne doit pas être exécuté pour régénérer les jeux de test déjà validés.

In [ ]:
"""Prépare un lot d'annotation et un audit des CSV extraits existants.

Usage : python Scripts/pipeline_et_validation/archive_mise_au_point/preparer_evaluation_extraction.ipynb
Le script ne modifie ni les PDF ni les CSV sources.
"""

from __future__ import annotations

import csv
import re
import unicodedata
from collections import Counter
from pathlib import Path


def trouver_racine() -> Path:
    courant = Path.cwd().resolve()
    for candidat in (courant, *courant.parents):
        if (candidat / "total_donnees").is_dir() and (candidat / "evaluation_extraction").is_dir():
            return candidat
    raise FileNotFoundError("Ouvre ce notebook depuis le dossier de travail.")

RACINE = trouver_racine()
SOURCE = RACINE / "total_donnees"
SORTIE = RACINE / "evaluation_extraction"
TEXTE = SORTIE / "textes"
ANNEES = range(2015, 2026)
COLONNES_REFERENCE = [
    "document_id", "page", "type_nomination", "date_conseil",
    "civilite", "nom", "matricule", "fonction", "ministere",
    "extrait_exact", "commentaire",
]


def normaliser(chaine: str) -> str:
    chaine = unicodedata.normalize("NFKD", chaine.casefold())
    chaine = "".join(c for c in chaine if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", chaine).strip()


def date_nom_fichier(chemin: Path) -> str:
    match = re.search(r"(20\d{2})_(\d{2})_(\d{2})", chemin.name)
    return f"{match[1]}-{match[2]}-{match[3]}" if match else ""


def choisir_documents() -> list[Path]:
    """Premier, médian et dernier PDF de chaque année, tous ceux de 2022."""
    pdfs = sorted(SOURCE.glob("*.pdf"))
    selection = []
    for annee in ANNEES:
        groupe = sorted(p for p in pdfs if date_nom_fichier(p).startswith(str(annee)))
        if not groupe:
            continue
        if annee == 2022:
            selection.extend(groupe)
        else:
            selection.extend(groupe[i] for i in sorted({0, len(groupe) // 2, len(groupe) - 1}))
    return selection


def lire_csv(chemin: Path) -> list[dict[str, str]]:
    if not chemin.exists():
        return []
    with chemin.open(newline="", encoding="utf-8-sig") as fichier:
        return list(csv.DictReader(fichier))


def ecrire_csv(chemin: Path, colonnes: list[str], lignes: list[dict[str, str]]) -> None:
    with chemin.open("w", newline="", encoding="utf-8-sig") as fichier:
        writer = csv.DictWriter(fichier, fieldnames=colonnes, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(lignes)


def decisions_existantes(chemin: Path, cle: tuple[str, ...]) -> dict[tuple[str, ...], dict[str, str]]:
    return {
        tuple(ligne.get(champ, "") for champ in cle): ligne
        for ligne in lire_csv(chemin)
    }


def debut_nominations(texte: str) -> int | None:
    # Évite les mentions « a procédé à des nominations » de l'introduction.
    motif = re.compile(r"(?im)^\s*(?:III|IV)\s*\.\s*NOMINATIONS\s*$")
    match = motif.search(texte)
    if match:
        return match.end()
    # La mise en page du PDF peut placer le titre sur une ligne plus longue.
    motif_souple = re.compile(r"(?i)\b(?:III|IV)\s*\.\s*NOMINATIONS\b")
    match = motif_souple.search(texte)
    return match.end() if match else None


def texte_pdf(chemin: Path) -> str:
    from pdfminer.high_level import extract_text

    texte = extract_text(str(chemin)) or ""
    if len(texte.strip()) >= 100:
        return texte
    # Certains PDF sont des images ; OCR page par page en français.
    from io import BytesIO

    import pymupdf
    import pytesseract
    from PIL import Image

    with pymupdf.open(chemin) as document:
        pages = []
        for page in document:
            pixmap = page.get_pixmap(matrix=pymupdf.Matrix(2, 2))
            image = Image.open(BytesIO(pixmap.tobytes("png")))
            pages.append(pytesseract.image_to_string(image, lang="fra"))
    return "\f".join(pages)


def audit_ligne(
    ligne: dict[str, str], numero: int, pdf: Path, section: str | None,
    compteur: Counter[tuple[str, str, str, str]],
) -> list[dict[str, str]]:
    nom = (ligne.get("nom") or "").strip()
    poste = (ligne.get("poste") or "").strip()
    ministere = (ligne.get("ministere") or "").strip()
    date = (ligne.get("date_conseil") or "").strip()[:10]
    cle = (date, normaliser(nom), normaliser(poste), normaliser(ministere))
    raisons = []
    if not nom or not poste:
        raisons.append("nom_ou_fonction_vide")
    if not ministere:
        raisons.append("ministere_vide")
    if date and date != date_nom_fichier(pdf):
        raisons.append("date_differente_du_nom_de_fichier")
    if compteur[cle] > 1:
        raisons.append("doublon_cle_provisoire")
    if section is None:
        raisons.append("section_nominations_non_detectee")
    elif nom and normaliser(nom) not in normaliser(section):
        raisons.append("nom_absent_de_la_section_nominations")
    return [
        {
            "document_id": pdf.name,
            "ligne_csv": str(numero),
            "raison": raison,
            "date_conseil": date,
            "nom": nom,
            "fonction": poste,
            "ministere": ministere,
            "decision_humaine": "",
            "correction": "",
        }
        for raison in raisons
    ]


def main() -> None:
    TEXTE.mkdir(parents=True, exist_ok=True)
    anciennes_predictions = decisions_existantes(
        SORTIE / "predictions_a_verifier.csv", ("document_id", "ligne_csv")
    )
    anciennes_alertes = decisions_existantes(
        SORTIE / "erreurs_a_verifier.csv", ("document_id", "ligne_csv", "raison")
    )
    documents = choisir_documents()
    inventaire = []
    alertes = []
    predictions = []

    for pdf in documents:
        csv_source = pdf.with_suffix(".csv")
        lignes = lire_csv(csv_source)
        erreur_lecture = ""
        try:
            texte = texte_pdf(pdf)
        except Exception as exc:
            texte = ""
            erreur_lecture = f"{type(exc).__name__}: {exc}"
        (TEXTE / f"{pdf.stem}.txt").write_text(texte, encoding="utf-8")
        position = debut_nominations(texte)
        section = texte[position:] if position is not None else None
        compteur = Counter(
            (
                (ligne.get("date_conseil") or "")[:10],
                normaliser(ligne.get("nom") or ""),
                normaliser(ligne.get("poste") or ""),
                normaliser(ligne.get("ministere") or ""),
            )
            for ligne in lignes
        )

        for numero, ligne in enumerate(lignes, start=2):
            alertes.extend(audit_ligne(ligne, numero, pdf, section, compteur))
            predictions.append(
                {
                    "document_id": pdf.name,
                    "ligne_csv": str(numero),
                    "date_conseil": (ligne.get("date_conseil") or "")[:10],
                    "civilite": ligne.get("civilite") or "",
                    "nom": ligne.get("nom") or "",
                    "matricule": ligne.get("matricule") or "",
                    "fonction": ligne.get("poste") or "",
                    "ministere": ligne.get("ministere") or "",
                    "decision_humaine": "",
                    "correction": "",
                }
            )

        inventaire.append(
            {
                "document_id": pdf.name,
                "annee": date_nom_fichier(pdf)[:4],
                "date_nom_fichier": date_nom_fichier(pdf),
                "pdf": str(pdf.relative_to(RACINE)),
                "csv_existant": str(csv_source.relative_to(RACINE)) if csv_source.exists() else "",
                "texte_pour_annotation": str((TEXTE / f"{pdf.stem}.txt").relative_to(RACINE)),
                "caracteres_texte": str(len(texte)),
                "rubrique_nominations_detectee": "oui" if position is not None else "non",
                "lignes_extraites": str(len(lignes)),
                "erreur_lecture": erreur_lecture,
                "statut_annotation": "a_faire",
            }
        )

    anciens_documents = decisions_existantes(SORTIE / "documents_echantillon.csv", ("document_id",))
    for ligne in inventaire:
        ancienne = anciens_documents.get((ligne["document_id"],), {})
        ligne["statut_annotation"] = ancienne.get("statut_annotation") or "a_faire"
    for ligne in predictions:
        ancienne = anciennes_predictions.get((ligne["document_id"], ligne["ligne_csv"]), {})
        ligne["decision_humaine"] = ancienne.get("decision_humaine", "")
        ligne["correction"] = ancienne.get("correction", "")
    for ligne in alertes:
        ancienne = anciennes_alertes.get((ligne["document_id"], ligne["ligne_csv"], ligne["raison"]), {})
        ligne["decision_humaine"] = ancienne.get("decision_humaine", "")
        ligne["correction"] = ancienne.get("correction", "")

    ecrire_csv(SORTIE / "documents_echantillon.csv", list(inventaire[0]), inventaire)
    ecrire_csv(SORTIE / "predictions_a_verifier.csv", list(predictions[0]), predictions)
    ecrire_csv(
        SORTIE / "erreurs_a_verifier.csv",
        ["document_id", "ligne_csv", "raison", "date_conseil", "nom", "fonction", "ministere", "decision_humaine", "correction"],
        alertes,
    )
    reference = SORTIE / "reference_annotations.csv"
    if not reference.exists():
        ecrire_csv(reference, COLONNES_REFERENCE, [])
    print(f"{len(documents)} documents ; {len(predictions)} prédictions ; {len(alertes)} alertes")
    print(f"Sortie : {SORTIE}")


## Exécution manuelle

Vérifier les chemins et les fichiers de sortie avant toute exécution.

In [ ]:
# main()